# Notebook 2 — Mathématiques & Statistiques pour le ML
### BNP Paribas — Senior AI Engineer

## Sommaire
1. [Algèbre Linéaire](#algebre)
2. [Calcul Différentiel & Gradient](#gradient)
3. [Probabilités & Bayes](#proba)
4. [Distributions](#distributions)
5. [Tests Statistiques](#tests)
6. [Théorie de l'Information](#info)
7. [Questions d'entretien](#questions)

---
## 1. Algèbre Linéaire

**Produit matriciel** : C = A @ B, O(n^3)
**SVD** : A = U*Sigma*V^T — PCA, compression, pseudo-inverse
**Valeurs propres** : A*v = lambda*v — PCA, spectral clustering
**Normes** : L1, L2, L-inf — régularisation, distances

| Décomposition | Formule | Usage |
|---|---|---|
| SVD | A = U*S*V^T | PCA, LSA, recommandation |
| Eigen | A*v = lambda*v | PCA, Spectral clustering |
| QR | A = Q*R | Moindres carrés, Gram-Schmidt |
| Cholesky | A = L*L^T | Sampling gaussien, kalman |

In [ ]:
import numpy as np

# ============================================================
# OPERATIONS FONDAMENTALES
# ============================================================
A = np.array([[1, 2, 3],[4, 5, 6],[7, 8, 9]], dtype=float)
B = np.array([[9, 8, 7],[6, 5, 4],[3, 2, 1]], dtype=float)

print("A @ B ="); print(A @ B)
print(f"Trace(A) = {np.trace(A)}")
print(f"det(A) = {np.linalg.det(A):.4f}  (0 => singuliere)")

# Resolution systeme Ax=b
A2 = np.array([[2.0, 1], [1, 3]])
b2 = np.array([5.0, 10])
x = np.linalg.solve(A2, b2)
print(f"\nSolution Ax=b: {x}")
print(f"Verification: {np.allclose(A2 @ x, b2)}")

In [ ]:
import numpy as np

# ============================================================
# SVD -- Singular Value Decomposition
# ============================================================
np.random.seed(42)
X = np.random.randn(6, 4)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
print(f"X: {X.shape}, U: {U.shape}, s: {s.shape}, Vt: {Vt.shape}")

# Reconstruction
X_rec = U @ np.diag(s) @ Vt
print(f"Erreur reconstruction: {np.linalg.norm(X - X_rec):.2e}")

# Low-rank approximation (rank-k)
for k in [1, 2, 3]:
    X_lr = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    var_explained = (s[:k]**2).sum() / (s**2).sum() * 100
    err = np.linalg.norm(X - X_lr)
    print(f"Rank-{k}: erreur={err:.4f}, variance={var_explained:.1f}%")

# ============================================================
# VALEURS PROPRES -- PCA
# ============================================================
print("\n=== PCA from scratch ===")
data = np.random.multivariate_normal([2,5], [[1,0.9],[0.9,2]], 300)
X_c = data - data.mean(axis=0)
cov = X_c.T @ X_c / (len(data)-1)
evals, evecs = np.linalg.eigh(cov)
idx = np.argsort(evals)[::-1]
evals, evecs = evals[idx], evecs[:, idx]
print(f"Variance PC1: {evals[0]/evals.sum()*100:.1f}%")
print(f"Variance PC2: {evals[1]/evals.sum()*100:.1f}%")

# Normes et distances
print("\n=== Normes ===")
v1, v2 = np.array([1.0,2.0,3.0]), np.array([4.0,5.0,6.0])
print(f"L1: {np.linalg.norm(v1,1):.3f}  L2: {np.linalg.norm(v1):.3f}  Linf: {np.linalg.norm(v1,np.inf):.3f}")
cos_sim = np.dot(v1,v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))
print(f"Cosine similarity: {cos_sim:.4f}  (1=identiques, 0=orthogonaux, -1=opposes)")

---
## 2. Calcul Différentiel & Gradient

**Gradient descent** : theta = theta - lr * nabla_theta L
**Règle de la chaîne** : dL/dx = dL/dy * dy/dx (backprop)
**Adam** : adapte le lr par paramètre avec moments du premier et deuxième ordre

| Optimiseur | Update | Avantage |
|---|---|---|
| SGD | w -= lr * grad | Simple, interprétable |
| Momentum | v = beta*v + grad; w -= lr*v | Réduit oscillations |
| RMSProp | w -= lr*grad / sqrt(EMA(grad^2)) | Bon pour RNN |
| Adam | Momentum + RMSProp + bias correction | Généralement meilleur |
| AdamW | Adam + weight decay correct | State of art |

In [ ]:
import numpy as np

# Gradient numerique vs analytique
def f(x):
    return x**4 - 4*x**2 + x

def df(x):
    return 4*x**3 - 8*x + 1

def numerical_grad(f, x, h=1e-5):
    return (f(x+h) - f(x-h)) / (2*h)

x = 2.0
print(f"Gradient analytique: {df(x):.8f}")
print(f"Gradient numerique:  {numerical_grad(f, x):.8f}")
print(f"Erreur: {abs(df(x)-numerical_grad(f,x)):.2e}")

# ============================================================
# ADAM FROM SCRATCH
# ============================================================
print("\n=== Adam Optimizer ===")
np.random.seed(42)
n = 100
X = np.random.randn(n)
y = 3*X + 5 + np.random.randn(n)*0.5

def adam(X, y, epochs=200, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    w, b = 0.0, 0.0
    mw, vw, mb, vb = 0.0, 0.0, 0.0, 0.0
    losses = []
    for t in range(1, epochs+1):
        yp = w*X + b
        loss = np.mean((y-yp)**2)
        losses.append(loss)
        dw = -2*np.mean((y-yp)*X)
        db = -2*np.mean(y-yp)
        # Update moments
        mw = beta1*mw + (1-beta1)*dw; vw = beta2*vw + (1-beta2)*dw**2
        mb = beta1*mb + (1-beta1)*db; vb = beta2*vb + (1-beta2)*db**2
        # Bias correction
        mw_h = mw/(1-beta1**t); vw_h = vw/(1-beta2**t)
        mb_h = mb/(1-beta1**t); vb_h = vb/(1-beta2**t)
        w -= lr * mw_h / (np.sqrt(vw_h)+eps)
        b -= lr * mb_h / (np.sqrt(vb_h)+eps)
    return w, b, losses

w_adam, b_adam, losses = adam(X, y)
print(f"Adam: w={w_adam:.4f} (true=3.0), b={b_adam:.4f} (true=5.0)")
print(f"Loss initial: {losses[0]:.4f}  final: {losses[-1]:.4f}")

---
## 3. Probabilités & Théorème de Bayes

**Bayes** : P(theta|D) = P(D|theta) * P(theta) / P(D)

**MLE** : theta_MLE = argmax P(D|theta)
**MAP** : theta_MAP = argmax P(D|theta) * P(theta)

> MAP avec prior N(0, 1/lambda) => Ridge (L2 regularization)
> MAP avec prior Laplace(0, 1/lambda) => Lasso (L1 regularization)

In [ ]:
import numpy as np
from scipy import stats

# ============================================================
# THEOREME DE BAYES -- Base Rate Fallacy
# ============================================================
print("=== Bayes: Detection de fraude ===")
p_fraud = 0.01
p_alerte_given_fraud = 0.95
p_alerte_given_normal = 0.05

p_alerte = p_alerte_given_fraud*p_fraud + p_alerte_given_normal*(1-p_fraud)
p_fraud_given_alerte = (p_alerte_given_fraud * p_fraud) / p_alerte

print(f"P(Fraude)         = {p_fraud:.3f}")
print(f"P(Alerte|Fraude)  = {p_alerte_given_fraud:.3f}")
print(f"P(Alerte|Normal)  = {p_alerte_given_normal:.3f}")
print(f"P(Alerte)         = {p_alerte:.4f}")
print(f"P(Fraude|Alerte)  = {p_fraud_given_alerte:.4f}")
print(f"=> Seulement {p_fraud_given_alerte:.1%} de chance d'etre une vraie fraude!")
print("   => Base rate fallacy: ignorer la freq. de base des evenements rares")

# ============================================================
# MLE -- Estimation des parametres
# ============================================================
print("\n=== MLE ===")
data = np.array([2.1, 2.5, 1.8, 2.3, 2.7, 1.9, 2.2, 2.4])
mu_mle = np.mean(data)
sigma_mle = np.std(data, ddof=0)  # ddof=0 pour MLE
log_lik = np.sum(stats.norm.logpdf(data, mu_mle, sigma_mle))
print(f"MLE mu={mu_mle:.4f}, sigma={sigma_mle:.4f}")
print(f"Log-vraisemblance: {log_lik:.4f}")

# MLE vs MAP demonstration (Ridge = MAP avec prior gaussien)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
np.random.seed(42)
X_d = np.random.randn(30, 8)
y_d = X_d[:,0]*2 + X_d[:,1]*0.5 + np.random.randn(30)*0.5

print("\n=== MLE vs MAP ===")
for name, model in [("MLE (OLS)", LinearRegression()),
                    ("MAP-L2 (Ridge)", Ridge(alpha=1.0)),
                    ("MAP-L1 (Lasso)", Lasso(alpha=0.1))]:
    m = model.fit(X_d, y_d)
    coef = m.coef_
    print(f"{name}: ||coef||_2={np.linalg.norm(coef):.3f}, zeros={np.sum(np.abs(coef)<0.01)}/8")

---
## 4. Distributions de Probabilité

| Distribution | E[X] | Var(X) | Usage |
|---|---|---|---|
| N(mu, sigma^2) | mu | sigma^2 | Init poids, résidus |
| Bernoulli(p) | p | p(1-p) | Classification binaire |
| Poisson(lambda) | lambda | lambda | Comptage |
| Beta(a,b) | a/(a+b) | - | Prior sur proba |
| Student-t(nu) | 0 | nu/(nu-2) | Tests, faible n |

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

print("=== Distribution Normale ===")
print(f"P(-1 < X < 1) = {stats.norm.cdf(1)-stats.norm.cdf(-1):.4f}  (68%)")
print(f"P(-2 < X < 2) = {stats.norm.cdf(2)-stats.norm.cdf(-2):.4f}  (95%)")
print(f"P(-3 < X < 3) = {stats.norm.cdf(3)-stats.norm.cdf(-3):.4f}  (99.7%)")
print(f"z(97.5%) = {stats.norm.ppf(0.975):.4f}  (=> IC 95% bilateral)")
print(f"z(99.5%) = {stats.norm.ppf(0.995):.4f}  (=> IC 99% bilateral)")

print("\n=== Theoreme Central Limite ===")
print("La moyenne de n variables i.i.d. -> N(mu, sigma^2/n)")
for n in [1, 5, 30, 100]:
    means = [np.mean(np.random.exponential(1, n)) for _ in range(10000)]
    print(f"  n={n:4d}: mean={np.mean(means):.4f}, std_obs={np.std(means):.4f}, std_theo={1/np.sqrt(n):.4f}")

print("\n=== Distributions continues ===")
for name, d in [("Normale N(0,1)", stats.norm()),
                ("t(5 dof)", stats.t(5)),
                ("t(30 dof)", stats.t(30)),
                ("Beta(2,5)", stats.beta(2,5))]:
    try:
        print(f"  {name}: mean={d.mean():.4f}, std={d.std():.4f}")
    except:
        print(f"  {name}: mean={d.mean():.4f}")

print("\n=== Distribution de Poisson ===")
for lam in [1, 5, 20]:
    d = stats.poisson(lam)
    print(f"  Poisson({lam}): E[X]=Var(X)={d.mean():.0f}  P(X={lam})={d.pmf(lam):.4f}")

---
## 5. Tests Statistiques

**Framework** :
1. Definir H0 (hypothese nulle) et H1
2. Choisir la statistique de test
3. Calculer la p-value : P(resultat aussi extreme | H0)
4. Decider : rejeter H0 si p-value < alpha (0.05)

| Type d'erreur | Definition | Consequence |
|---|---|---|
| Type I (alpha) | Rejeter H0 alors qu'elle est vraie | Fausse alarme |
| Type II (beta) | Accepter H0 alors qu'elle est fausse | Manque l'effet |
| Puissance | 1 - beta | Capacite a detecter un vrai effet |

In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

# ============================================================
# 1. TEST T DE STUDENT (comparer 2 modeles)
# ============================================================
print("=== Test t de Welch (independant) ===")
A = np.array([0.82, 0.85, 0.79, 0.88, 0.83, 0.81, 0.86, 0.84, 0.87, 0.85])
B = np.array([0.78, 0.80, 0.82, 0.79, 0.77, 0.81, 0.78, 0.80, 0.79, 0.78])

t, p = stats.ttest_ind(A, B, equal_var=False)
print(f"A: mean={A.mean():.4f}, B: mean={B.mean():.4f}")
print(f"t={t:.4f}, p={p:.6f}")
print(f"=> {'Difference significative' if p<0.05 else 'Pas de difference'} (alpha=0.05)")

# 2. CHI-2
print("\n=== Test Chi-2 d'independance ===")
observed = np.array([[55, 45],[35, 65]])
chi2, p_c, dof, expected = stats.chi2_contingency(observed)
print(f"chi2={chi2:.4f}, dof={dof}, p={p_c:.4f}")
print(f"=> {'Dependance significative' if p_c<0.05 else 'Independants'}")

# 3. SHAPIRO-WILK (normalite)
print("\n=== Test Shapiro-Wilk (normalite) ===")
for name, data in [("Normale", np.random.normal(0,1,50)), ("Exponentielle", np.random.exponential(1,50))]:
    stat, p_sw = stats.shapiro(data)
    print(f"  {name}: stat={stat:.4f}, p={p_sw:.4f} => {'Normale' if p_sw>0.05 else 'Non-normale'}")

# 4. MANN-WHITNEY U (non-parametrique)
print("\n=== Mann-Whitney U (non-parametrique) ===")
u, p_u = stats.mannwhitneyu(A, B, alternative="two-sided")
print(f"U={u}, p={p_u:.6f}")

# 5. TESTS MULTIPLES
print("\n=== Probleme des tests multiples ===")
k, alpha = 20, 0.05
fwer = 1 - (1-alpha)**k
print(f"k={k} tests, alpha={alpha}")
print(f"P(au moins 1 faux positif) = {fwer:.4f}")
print(f"Bonferroni: alpha_corrige = {alpha/k:.4f}")
print("BH (Benjamini-Hochberg): controle FDR, moins conservateur")

# 6. INTERVALLES DE CONFIANCE
print("\n=== Intervalles de confiance ===")
n, mean, se = len(A), A.mean(), stats.sem(A)
for conf in [0.90, 0.95, 0.99]:
    lo, hi = stats.t.interval(conf, df=n-1, loc=mean, scale=se)
    print(f"  {int(conf*100)}% CI: [{lo:.4f}, {hi:.4f}]")

---
## 6. Théorie de l'Information

- **Entropie** H(X) = -sum p*log(p) : incertitude
- **Cross-entropie** H(P,Q) = -sum p*log(q) : loss classification
- **KL divergence** KL(P||Q) = sum p*log(p/q) >= 0
- **Relation** : H(P,Q) = H(P) + KL(P||Q)
- **Gini impurity** = 1 - sum p_i^2 : alternative à l'entropie pour les arbres

> **VAE** : ELBO = E[log p(x|z)] - KL(q(z|x)||p(z))
> **RLHF** : Ajout d'une pénalité KL pour ne pas trop s'éloigner du modèle de référence

In [ ]:
import numpy as np

def entropy(p):
    p = np.array(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log(p + 1e-15))

def cross_entropy(p_true, q_pred):
    q = np.clip(q_pred, 1e-10, 1.0)
    return -np.sum(np.array(p_true, dtype=float) * np.log(q))

def kl_divergence(p, q):
    p, q = np.array(p, float), np.clip(q, 1e-10, 1.0)
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask]/q[mask]))

def gini(p):
    return 1 - sum(pi**2 for pi in p)

print("=== Entropie ===")
for name, d in [("Uniforme", [.25,.25,.25,.25]),
                ("Quasi-certain", [.97,.01,.01,.01]),
                ("Bimodale", [.45,.45,.05,.05])]:
    print(f"  H({name}): {entropy(d):.4f} nats")

print("\n=== Cross-Entropie (loss classification) ===")
y_true = [1.0, 0.0, 0.0]
for name, pred in [("Bonne", [.9,.07,.03]),
                   ("Mauvaise", [.1,.6,.3]),
                   ("Tres mauvaise", [.01,.01,.98])]:
    print(f"  {name}: CE={cross_entropy(y_true, pred):.4f}")

print("\n=== KL Divergence ===")
y_good = [.9,.07,.03]
y_bad = [.1,.6,.3]
print(f"  KL(true||good): {kl_divergence(y_true, y_good):.4f}")
print(f"  KL(true||bad):  {kl_divergence(y_true, y_bad):.4f}")
print(f"  KL(P||P)=0: {kl_divergence(y_true, y_true):.8f}")
print("  KL n est PAS symetrique: KL(P||Q) != KL(Q||P)")
print(f"  KL(good||true): {kl_divergence(y_good, [max(v,.01) for v in y_true]):.4f}")

print("\n=== Entropie vs Gini (arbres de decision) ===")
for p in [0.5, 0.7, 0.9, 1.0]:
    d = [p, 1-p]
    print(f"  p={p}: H={entropy(d):.4f}, Gini={gini(d):.4f}")

---
## 7. Questions d'entretien fréquentes

**Q: MLE vs MAP, différence concrète ?**
> MLE = moindres carrés (regression lineaire). MAP avec prior gaussien = Ridge. MAP avec prior laplacien = Lasso. La regularisation est une forme de prior bayesien.

**Q: Pourquoi normaliser les features ?**
> 1. Gradient descent converge plus vite (iso-contours spheriques). 2. Algorithmes bases sur distances (KNN, SVM, K-Means) sont tres sensibles a l'echelle. 3. Meilleure condition numerique.

**Q: Biais-variance tradeoff ?**
> Erreur totale = Biais^2 + Variance + Bruit irreductible. Biais = erreur systematique (underfitting). Variance = sensibilite aux donnees (overfitting). Ensemble methods (bagging) reduisent la variance. Boosting reduit le biais.

**Q: Qu'est-ce que la colinearite ?**
> Features fortement correlees. X^TX devient singuliere => instabilite numerique. Coefficients instables et difficiles a interpreter. Solutions: Ridge, PCA, VIF pour detecter, supprimer des features.

**Q: Ou est utilisee la KL divergence ?**
> VAE (ELBO), RLHF (penalite KL pour rester proche du modele de reference), distillation de modeles, variational inference, information bottleneck.

**Q: Quelle est la propriete cle de la distribution de Poisson ?**
> E[X] = Var(X) = lambda. Si Var > Mean, on a de la sur-dispersion (utiliser negative binomiale). Usage: comptage d'evenements rares (fraudes, clics).
